# Step 07: Fine-tune Qwen2.5-3B with QLoRA (Kaggle)

**What this notebook does:** teaches `Qwen2.5-3B-Instruct` to answer our ticket-triage prompt the way
the teacher (`gpt-4.1`) does, then answers the 200 gold tickets so we can score it back on your laptop.

**Before you run it (once):**
1. On your laptop: `uv run python scripts/07_prepare_training_data.py` (already done, makes `data/training/`).
2. kaggle.com -> Datasets -> New Dataset -> upload the 3 files in `data/training/`
   (`train.jsonl`, `val.jsonl`, `gold_prompts.jsonl`). Name it `ticket-triage-training`. Keep it **private**.
3. kaggle.com -> Code -> New Notebook -> File -> Import Notebook -> this file.
4. Right panel: **Add Input** -> your dataset. **Session options -> Accelerator: GPU T4 x2**
   (we only use one of them), **Internet: On** (to download the model and libraries).
5. **Run All.** It takes roughly 30-60 minutes. Nothing here uses your laptop.

**After it finishes:** download `ft-qwen-kaggle.jsonl` and `qwen-triage-lora.zip` from the Output tab.

### The idea in plain words
- The base model already knows English and can write JSON. It does not know *our* rules
  (what counts as `customer_service`, how to judge `priority`).
- **LoRA:** instead of changing all 3 billion numbers in the model, we freeze them and add small
  extra "adapter" layers, then train only those (about 1% of the size).
- **QLoRA:** the frozen model is also squeezed to 4-bit numbers so it fits in the T4's 16 GB.
- We show it 1,540 examples of `ticket -> teacher's JSON answer`. It guesses, we measure how
  wrong it was (the **loss**), and nudge the adapter to be less wrong. Repeat.

In [ ]:
# Install the training libraries (Kaggle already has torch). Takes about a minute.
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
import glob
import json
import math
import os
import shutil
import time

import torch
import transformers
import peft
import trl

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Settings
Every number you might want to change is here. Defaults are the standard, safe QLoRA recipe.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen-triage-lora"
SEED = 42

# LoRA: how big the trainable add-on is. r = its "width". Bigger r = more capacity, more memory.
LORA_R = 16
LORA_ALPHA = 32          # scaling; keeping alpha = 2 x r is a common default
LORA_DROPOUT = 0.05      # randomly ignores 5% of the adapter while training, to reduce overfitting

# Training
EPOCHS = 3               # how many times it sees the whole training set
LEARNING_RATE = 2e-4     # step size; 2e-4 is the usual QLoRA value
BATCH_SIZE = 4           # examples per step on the GPU
GRAD_ACCUM = 4           # collect 4 steps before updating: effective batch = 4 x 4 = 16
MAX_LENGTH = 1280        # longest example in tokens (ours are about 950 at most)

In [ ]:
# Find the files from the dataset you attached (the folder name depends on what you called it).
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found. Did you add the dataset as an input to this notebook?"
    return hits[0]

TRAIN_FILE = find("train.jsonl")
VAL_FILE = find("val.jsonl")
GOLD_FILE = find("gold_prompts.jsonl")
print(TRAIN_FILE, VAL_FILE, GOLD_FILE, sep="\n")

## 2. Look at the data
Always read one example before training on 1,500 of them.

In [ ]:
from datasets import load_dataset

train_data = load_dataset("json", data_files=TRAIN_FILE, split="train")
val_data = load_dataset("json", data_files=VAL_FILE, split="train")
print(len(train_data), "training examples,", len(val_data), "validation examples")

example = train_data[0]
print("--- ticket the model sees (user message) ---")
print(example["prompt"][1]["content"][:400])
print("--- answer it must learn to write ---")
print(example["completion"][0]["content"])

# 'id' and 'category' were only kept for bookkeeping; the trainer needs just prompt + completion.
train_data = train_data.remove_columns(["id", "category"])
val_data = val_data.remove_columns(["id", "category"])

## 3. Load the model in 4-bit and the tokenizer
The tokenizer turns text into numbers (tokens). `Qwen`'s chat template wraps our messages in the
special markers the model was built to expect.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Check the longest example fits MAX_LENGTH, otherwise the end of the JSON would be cut off.
longest = 0
for row in train_data:
    text = tokenizer.apply_chat_template(row["prompt"] + row["completion"], tokenize=False)
    longest = max(longest, len(tokenizer(text)["input_ids"]))
print("longest training example:", longest, "tokens (limit", MAX_LENGTH, ")")
assert longest < MAX_LENGTH, "raise MAX_LENGTH"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # squeeze the frozen weights to 4 bits each
    bnb_4bit_quant_type="nf4",            # a 4-bit format designed for neural-network weights
    bnb_4bit_use_double_quant=True,       # squeeze the squeezing constants too
    bnb_4bit_compute_dtype=torch.float16, # do the maths in fp16 (the T4 has no bf16)
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},                   # everything on the first GPU
    torch_dtype=torch.float16,
)
model.config.use_cache = False            # the cache is only for generation, and conflicts with checkpointing
print("model loaded, memory used GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

## 4. Add the LoRA adapter and train
`target_modules` lists which layers get an adapter: the attention projections (q, k, v, o) and the
feed-forward projections (gate, up, down). Training all of them works better than attention alone.

`SFTTrainer` (supervised fine-tuning) builds the batches for us. Because our data has a `prompt`
and a `completion`, it only measures the loss on the **completion**: the model is graded on the JSON
it writes, not on repeating the long instructions.

Watch two numbers: **loss** (training error) and **eval_loss** (error on 300 tickets it never trains
on). Both should fall. If loss keeps falling but eval_loss rises, the model is memorising (overfitting).

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",           # learning rate glides down to ~0 by the end
    warmup_ratio=0.05,                    # ...after a short gentle start
    optim="paged_adamw_8bit",             # memory-light optimiser
    fp16=True,
    gradient_checkpointing=True,          # trade some speed for much less memory
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LENGTH,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
started = time.time()
result = trainer.train()
print(f"training took {(time.time() - started) / 60:.1f} minutes")
print(result.metrics)

## 5. Did it learn? Loss curves

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps = [h["step"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]
eval_steps = [h["step"] for h in history if "eval_loss" in h]
eval_loss = [h["eval_loss"] for h in history if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(train_steps, train_loss, label="training loss")
plt.plot(eval_steps, eval_loss, "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print("first eval_loss:", round(eval_loss[0], 3), "| last eval_loss:", round(eval_loss[-1], 3), "| best:", round(min(eval_loss), 3))

In [ ]:
# Save only the small adapter (tens of MB), plus a zip to download.
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
shutil.make_archive("/kaggle/working/qwen-triage-lora", "zip", OUTPUT_DIR)
print("adapter size MB:", round(os.path.getsize("/kaggle/working/qwen-triage-lora.zip") / 1e6, 1))

## 6. Answer the 200 gold tickets
The prompts contain **no labels**. We only collect the model's raw answers in the same file format
as `evals/run_model.py`, so your laptop can score them with `evals/score.py`, exactly like the
baselines. Greedy decoding (`do_sample=False`) makes the answers repeatable.

(The latency saved here is for a Kaggle T4 with batching, so it is NOT comparable to your laptop.
Real latency is measured in Steps 10 and 12.)

In [ ]:
model = trainer.model
model.eval()
model.config.use_cache = True
tokenizer.padding_side = "left"           # generation needs the padding on the left

gold = [json.loads(line) for line in open(GOLD_FILE, encoding="utf-8")]
OUT_FILE = "/kaggle/working/ft-qwen-kaggle.jsonl"
BATCH = 8

with open(OUT_FILE, "w", encoding="utf-8") as out:
    for start in range(0, len(gold), BATCH):
        batch = gold[start:start + BATCH]
        prompts = [tokenizer.apply_chat_template(g["messages"], tokenize=False, add_generation_prompt=True) for g in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        t0 = time.time()
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=450, do_sample=False,
                                       pad_token_id=tokenizer.pad_token_id)
        seconds = (time.time() - t0) / len(batch)
        for g, ids in zip(batch, generated):
            new_tokens = ids[inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
            out.write(json.dumps({"id": g["id"], "raw": raw, "latency_s": round(seconds, 3),
                                  "output_tokens": int((new_tokens != tokenizer.pad_token_id).sum()),
                                  "model": "qwen2.5-3b-qlora", "format": "none"}, ensure_ascii=False) + "\n")
        print(f"{min(start + BATCH, len(gold))}/{len(gold)}")

print("saved", OUT_FILE)

In [ ]:
# Quick look: how many answers are valid JSON with the right keys? (Real scoring happens on your laptop.)
good = 0
for line in open(OUT_FILE, encoding="utf-8"):
    row = json.loads(line)
    try:
        data = json.loads(row["raw"])
        if {"category", "priority", "sentiment", "products", "summary", "suggested_reply"} <= set(data):
            good += 1
    except Exception:
        pass
print(good, "of", len(gold), "answers parse as JSON with all six fields")

## 7. Next (on your laptop)
1. Output tab -> download **`ft-qwen-kaggle.jsonl`** and **`qwen-triage-lora.zip`** (and `loss_curve.png`).
2. Put `ft-qwen-kaggle.jsonl` in `evals/runs/` and run:
   `uv run python evals/score.py --run ft-qwen-kaggle`
3. Compare with the baseline (`evals/reports/base-qwen-schema.md`). That is Step 08.
4. Keep the adapter zip: Step 09 merges it into the model and converts it to GGUF for Ollama.